In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.nn import functional as F
import numpy as np
import pandas as pd
from generate_data import *
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from tqdm import tqdm

import pickle
from detect import detect as detect_1
from detect import detect_group7 as detect_7

from matplotlib.backends.backend_pdf import PdfPages

#### all of settings should be same as training part 1


In [3]:
## label to group mapping

label_group_dict = {
    # label: group
    0:4,
    1:6,
    2:2,
    3:3,
    4:5,
    5:7
}

In [4]:
## define model

class LSTMSignalClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])  # Take the output from the last time step
        return out

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_size = 1
hidden_size = 128
num_layers = 3
num_classes = 6
model = LSTMSignalClassifier(input_size, hidden_size, num_layers, num_classes)
model.load_state_dict(torch.load('model/best.pt')) ## replace this to real model path
model.to(device)
model.eval()

LSTMSignalClassifier(
  (lstm): LSTM(1, 128, num_layers=3, batch_first=True)
  (fc): Linear(in_features=128, out_features=6, bias=True)
)

In [6]:
## read sample pos data
with open('sample_pos.pkl', 'rb') as f:
    signals_pos = pickle.load(f)

len(signals_pos)

5639

In [7]:
signals_pos

[array([[ 1.,  4.],
        [ 2., 25.],
        [ 3., 32.],
        [ 4., 35.],
        [ 5., 43.],
        [ 6., 43.],
        [ 7., 43.],
        [ 8., 43.],
        [ 9., 46.],
        [10., 46.],
        [11., 46.],
        [12., 46.],
        [13., 46.],
        [14., 48.],
        [15., 51.],
        [16., 51.],
        [17., 54.],
        [18., 54.],
        [19., 51.],
        [20., 54.],
        [21., 60.],
        [22., 60.],
        [23., 62.],
        [24., 62.],
        [25., 62.],
        [26., 57.],
        [27., 57.],
        [28., 57.],
        [29., 57.],
        [30., 57.],
        [31., 57.],
        [32., 54.],
        [33., 54.],
        [34., 57.],
        [35., 57.],
        [36., 60.],
        [37., 63.],
        [38., 65.],
        [39., 71.],
        [40., 77.],
        [41., 80.],
        [42., 80.],
        [43., 80.],
        [44., 80.],
        [45., 80.],
        [46., 80.],
        [47., 83.]]),
 array([[   1., -173.],
        [   2., -184.],
        [ 

In [9]:
## for pos, first filter group 1 and 100 
group100 = []
groups = []

for signal in tqdm(signals_pos):
    pos = signal[:, 1]
    x = np.arange(len(pos))
    y = pos[:]
    x = x * 2000  ## ensure units of x and y are aligned 
    group = detect_1(x, y, method='pca')
    groups.append(group)
    if group == 100:
        group100.append(signal)

100%|██████████| 5639/5639 [00:00<00:00, 14856.33it/s]


In [10]:
## for group 100,
## use ML model to predict group

p_thres = 0.75
groups = []
idx_group = []
for i, each in tqdm(enumerate(group100)):
    signal = each[:, 1]
    x = normalize1d(resample(signal, 100))
    x_tensor = torch.tensor(x.reshape(100,1), dtype=torch.float32).unsqueeze(0).to(device)
    y_hat = model(x_tensor)
    p = F.softmax(y_hat, dim=1)
    c = np.array(p.tolist()).reshape(-1).argmax()
    p_max = np.max(np.array(p.tolist()).reshape(-1))
    group = 100
    if p_max > p_thres:
        group = label_group_dict[c]
    groups.append(group)
    idx_group.append((i, group))

2220it [00:10, 205.47it/s]


In [11]:
np.unique(groups, return_counts=True)

(array([  2,   3,   4,   5,   6,   7, 100]),
 array([533, 188, 161, 248, 329,  70, 691], dtype=int64))

## Select data for business labeling
## output as pdf

In [12]:
np_idx_group = np.array(idx_group)
idx, group = np_idx_group[:,0], np_idx_group[:, 1]

In [13]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
for i, (train_index, test_index) in enumerate(sss.split(idx, group)):
    break

In [14]:
idx_group_sorted = sorted(np_idx_group[test_index].tolist(), key=lambda x: (x[-1], x[0]))

In [15]:
idx_signal_dict = {}
for i, g in tqdm(sorted(np_idx_group[test_index].tolist(), key=lambda x: (x[-1], x[0]))):
    signal = group100[i][:, 1]
    idx_signal_dict[i] = signal

100%|██████████| 444/444 [00:00<00:00, 445093.45it/s]


In [ ]:
# Create the PdfPages object to which we will save the pages:
# The with statement makes sure that the PdfPages object is closed properly at
# the end of the block, even if an Exception occurs.

import matplotlib.pyplot as plt

with PdfPages('pos_labeling_new.pdf') as pdf:
    page = 0
    for i, g in tqdm(sorted(np_idx_group[test_index].tolist(), key=lambda x: (x[-1], x[0]))):
        page += 1
        fig = plt.figure(figsize=(6.4, 4.8))
        signal = group100[i][:, 1]
        plt.plot(signal)

        ## vertical line
        for x in range(0, len(signal), 5):
            plt.plot([x, x], [min(signal), signal[x]], color='gray', linestyle='--')

        plt.title(f'ID: {i}\nPredicted: Group {g}')
        fig.text(0.5, 0.02, f'Page {page}', ha='center', fontsize=10)
        plt.xticks(np.arange(0, len(signal), 5))


        pdf.savefig()  # saves the current figure into a pdf page
        plt.close()

100%|██████████| 444/444 [01:28<00:00,  5.00it/s]
